In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype(int)

X = X / 255.0

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Number of classes:", len(np.unique(y_train)))

print("First 5 labels:", y_train[:5])
print("First 5 images:")
for i in range(5):
    print(X_train[i].reshape(28, 28))

n_input = 784
n_hidden = 128
n_output = 10

# Initialize weights with small random values
# Initialize biases with zeros
W1 = np.random.randn(n_hidden, n_input) * 0.01
b1 = np.zeros(n_hidden)
W2 = np.random.randn(n_output, n_hidden) * 0.01
b2 = np.zeros(n_output)

X_train shape: (56000, 784)
X_test shape: (14000, 784)
Number of classes: 10
First 5 labels: [5 4 8 0 2]
First 5 images:
[[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.   

In [2]:
def relu(x):
    return np.maximum(0, x)


def softmax(x):
    exp_z = np.exp(x - np.max(x))
    return exp_z / np.sum(exp_z)


def forward(x, W1, b1, W2, b2):
    # layer 1
    z1 = np.dot(W1, x) + b1
    a1 = relu(z1)

    # layer 2
    z2 = np.dot(W2, a1) + b2
    a2  = softmax(z2)

    return z1, a1, z2, a2

x_sample = X_train[0]
z1, a1, z2, a2 = forward(x_sample, W1, b1, W2, b2)

print("z1 shape:", z1.shape)
print("a1 shape:", a1.shape)
print("z2 shape:", z2.shape)
print("a2 shape:", a2.shape)
print("Output probabilities:", a2)
print("Sum of probabilities:", np.sum(a2))

z1 shape: (128,)
a1 shape: (128,)
z2 shape: (10,)
a2 shape: (10,)
Output probabilities: [0.09958851 0.1005514  0.10031734 0.10033681 0.09938826 0.0996777
 0.09990707 0.09955163 0.10096637 0.09971491]
Sum of probabilities: 1.0


In [3]:

def one_hot(y, n_classes=10):
    result = np.zeros(n_classes)
    result[y] = 1
    return result

def cross_entropy_loss(y_true, y_pred):
    # clip to avoid log(0)
    y_pred_clipped = np.clip(y_pred, 1e-10, 1.0)
    return -np.sum(y_true * np.log(y_pred_clipped))

# Test
y_sample = y_train[0]
y_onehot = one_hot(y_sample)

z1, a1, z2, a2 = forward(x_sample, W1, b1, W2, b2)
loss = cross_entropy_loss(y_onehot, a2)

print("True label:", y_sample)
print("One-hot:", y_onehot)
print("Loss:", loss)
print("Expected loss for random model: ~", -np.log(0.1))

True label: 5
One-hot: [0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
Loss: 2.3058133122948097
Expected loss for random model: ~ 2.3025850929940455


In [4]:

def relu_derivative(z):
    return np.where(z > 0, 1, 0)

def backward(x, y_onehot, z1, a1, z2, a2, w1, w2):
    # Step 1: gradient at output layer
    dz2 = a2 - y_onehot

    # Step 2 & 3: gradients for W2, b2
    dw2 = np.outer(dz2, a1)
    db2 = dz2

    # Step 4: propagate back to hidden layer
    da1 = np.dot(w2.T, dz2)

    # Step 5: through ReLU
    dz1 = da1 * relu_derivative(z1)

    # Step 6 & 7: gradients for W1, b1
    dw1 = np.outer(dz1, x)
    db1 = dz1

    return dw1, db1, dw2, db2

dW1, db1, dW2, db2 = backward(x_sample, y_onehot, z1, a1, z2, a2, W1, W2)

print("dW1 shape:", dW1.shape)
print("dW2 shape:", dW2.shape)

dW1 shape: (128, 784)
dW2 shape: (10, 128)


In [5]:

def train(x_train, y_train, w1, b1, w2, b2, learning_rate=0.01, epochs=10):
    for epoch in range(epochs):
        total_loss = 0
        for i in range(len(x_train)):
            # forward pass
            x = x_train[i]
            y = one_hot(y_train[i])
            z1, a1, z2, a2 = forward(x, w1, b1, w2, b2)

            # compute loss
            loss = cross_entropy_loss(y, a2)
            total_loss += loss

            # backward pass
            dw1, db1, dw2, db2 = backward(x, y, z1, a1, z2, a2, w1, w2)

            w1 = w1 - learning_rate * dw1
            b1 = b1 - learning_rate * db1
            w2 = w2 - learning_rate * dw2
            b2 = b2 - learning_rate * db2

        avg_loss = total_loss / len(x_train)
        print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

    return w1, b1, w2, b2

# Train — chạy 3 epochs trước để xem loss giảm không
W1, b1, W2, b2 = train(X_train, y_train, W1, b1, W2, b2, learning_rate=0.01, epochs=3)

Epoch 1, Loss: 0.2655
Epoch 2, Loss: 0.1088
Epoch 3, Loss: 0.0753


In [7]:
def predict(x, w1, b1, w2, b2):
    z1, a1, z2, a2 = forward(x, w1, b1, w2, b2)
    return np.argmax(a2)

correct = 0
for i in range(len(X_test)):
    y_pred = predict(X_test[i], W1, b1, W2, b2)
    if y_pred == y_test[i]:
        correct = correct + 1

accuracy = correct / len(X_test) * 100
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 96.81%
